# Thesis figures

River data, station map and ALADIN figures used in the thesis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# primer H in Q za kobarid - 8080
postaja = 4230


path = f"data/reke/POV/POV/{postaja}"

h = pd.read_csv(path+f'/H/urne/2022/{postaja}_H_urne_2022.csv', sep=";")
q = pd.read_csv(path+f'/Q/urne/2022/{postaja}_Q_urne_2022.csv', sep=";")
h['datum'] = pd.to_datetime(h['datum'])
h = h.set_index('datum').sort_index().rename(columns={'vrednost':f'H'})
q['datum'] = pd.to_datetime(q['datum'])
q = q.set_index('datum').sort_index().rename(columns={'vrednost':f'Q'})
None

In [ ]:
# select one month
h = h.loc['2022-11-03':'2022-11-08']
q = q.loc['2022-11-03':'2022-11-08']

fig, ax1 = plt.subplots()
ax1.set_xlabel('Datum', fontsize=14)
ax1.set_ylabel('Višina vodostaja (cm)', color='blue', fontsize=14)
ax1.plot(h.index, h['H'], color='blue')
ax1.tick_params(axis='y', labelcolor='blue', labelsize=12)
ax1.set_title(f"Pretok in višina vodostaja pri postaji Zminec na Poljanski Sori", fontsize=16)
plt.xticks(rotation=45, fontsize=12)
ax1.tick_params(axis='x', labelsize=12)

ax2 = ax1.twinx()
ax2.set_ylabel('Pretok (m$^3$/s)', color='red', fontsize=14)
ax2.plot(q.index, q['Q'], color='red')
ax2.tick_params(axis='y', labelcolor='red', labelsize=12)



In [ ]:
import hidro_and_meteo_helper_functions as sf 
#sf.get_Q_df_for_station(3660).loc['2014-01-01':'2018-01-01'].plot()
sf.get_station_and_river_name(3660)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# First station data (Litija na Savi)
postaja = 3650
years = [2012, 2013, 2014]

path = f"data/reke/POV/POV/{postaja}"

# Initialize empty dataframes
h_all_1 = pd.DataFrame()
q_all_1 = pd.DataFrame()

# Loop through years
for year in years:
    # Read H data
    h_year = pd.read_csv(path+f'/H/dnevne/{year}/{postaja}_H_dnevne_{year}.csv', sep=";")
    h_year['datum'] = pd.to_datetime(h_year['datum'])
    h_year = h_year.set_index('datum').sort_index().rename(columns={'vrednost':f'Višina vodostaja'})
    
    # Read Q data
    q_year = pd.read_csv(path+f'/Q/dnevne/{year}/{postaja}_Q_dnevne_{year}.csv', sep=";")
    q_year['datum'] = pd.to_datetime(q_year['datum'])
    q_year = q_year.set_index('datum').sort_index().rename(columns={'vrednost':f'Pretok'})
    
    # Append to the combined dataframes
    h_all_1 = pd.concat([h_all_1, h_year])
    q_all_1 = pd.concat([q_all_1, q_year])

# Sort by date
h1 = h_all_1.sort_index()
q1 = q_all_1.sort_index()

# Second station data (Zminec na Poljanski Sori)
postaja = 3660

# Create empty dataframes to store data from multiple years
h_all_2 = pd.DataFrame()
q_all_2 = pd.DataFrame()

# Read data for years 2016, 2017, 2018
for year in [2016, 2017, 2018]:
    path = f"data/reke/POV/POV/{postaja}"
    
    # Read H data
    h_year = pd.read_csv(path+f'/H/dnevne/{year}/{postaja}_H_dnevne_{year}.csv', sep=";")
    h_year['datum'] = pd.to_datetime(h_year['datum'])
    h_year = h_year.set_index('datum').sort_index().rename(columns={'vrednost':f'Višina vodostaja'})
    
    # Read Q data
    q_year = pd.read_csv(path+f'/Q/dnevne/{year}/{postaja}_Q_dnevne_{year}.csv', sep=";")
    q_year['datum'] = pd.to_datetime(q_year['datum'])
    q_year = q_year.set_index('datum').sort_index().rename(columns={'vrednost':f'Pretok'})
    
    # Append to the combined dataframes
    h_all_2 = pd.concat([h_all_2, h_year])
    q_all_2 = pd.concat([q_all_2, q_year])

# Sort by date
h2 = h_all_2.sort_index()
q2 = q_all_2.sort_index()

# Resample data to weekly frequency
h1_weekly = h1.resample('W').mean()
q1_weekly = q1.resample('W').mean()
h2_weekly = h2.resample('W').mean()
q2_weekly = q2.resample('W').mean()

# Find global min/max for Višina vodostaja and Pretok for y-axis limits (same scale for both graphs)
h_min = min(h1_weekly['Višina vodostaja'].min(), h2_weekly['Višina vodostaja'].min())
h_max = max(h1_weekly['Višina vodostaja'].max(), h2_weekly['Višina vodostaja'].max())
q_min = min(q1_weekly['Pretok'].min(), q2_weekly['Pretok'].min())
q_max = max(q1_weekly['Pretok'].max(), q2_weekly['Pretok'].max())

# Add a small margin to the limits for better visualization
h_margin = 0.05 * (h_max - h_min) if h_max > h_min else 1
q_margin = 0.05 * (q_max - q_min) if q_max > q_min else 1
h_ylim = (h_min - h_margin, h_max + h_margin)
q_ylim = (q_min - q_margin, q_max + q_margin)

# Create a figure with two subplots side by side
fig = plt.figure(figsize=(14, 6))  # Reference: <Figure size 1400x600 with 4 Axes>

# Define font sizes (reference: file_context_0)
label_fontsize = 14
title_fontsize = 16
tick_fontsize = 12
legend_fontsize = 12

# First subplot - Litija na Savi
ax1 = fig.add_subplot(1, 2, 1)
ax1.set_xlabel('Datum', fontsize=label_fontsize)
ax1.set_ylabel('Višina vodostaja (cm)', color='blue', fontsize=label_fontsize)
ax1.set_ylim(h_ylim)

ax1.plot(h1_weekly.index, h1_weekly['Višina vodostaja'], color='blue')
# Add mean line for Višina vodostaja
h1_mean = h1['Višina vodostaja'].mean()
ax1.axhline(y=h1_mean, color='blue', linestyle='--', alpha=0.7, label=f'Povprečna višina vodostaja: {h1_mean:.2f} cm')
ax1.tick_params(axis='y', labelcolor='blue', labelsize=tick_fontsize)
ax1.set_title(f"Povprečni tedenski pretok in višina vodostaja\npri postaji Litija na Savi - pred prenovo", fontsize=title_fontsize)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, fontsize=tick_fontsize)
ax1.tick_params(axis='x', labelsize=tick_fontsize)

ax1_twin = ax1.twinx()
ax1_twin.set_ylabel('Pretok (m$^3$/s)', color='red', fontsize=label_fontsize)
ax1_twin.set_ylim(q_ylim)
ax1_twin.plot(q1_weekly.index, q1_weekly['Pretok'], color='red')
# Add mean line for Pretok
q1_mean = q1['Pretok'].mean()
ax1_twin.axhline(y=q1_mean, color='red', linestyle='--', alpha=0.7, label=f'Povprečni pretok: {q1_mean:.2f} m³/s')
ax1_twin.tick_params(axis='y', labelcolor='red', labelsize=tick_fontsize)

# Add legends for both axes
# Combine both legends into one
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=legend_fontsize)

# Second subplot - Zminec na Poljanski Sori
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_xlabel('Datum', fontsize=label_fontsize)
ax2.set_ylabel('Višina vodostaja (cm)', color='blue', fontsize=label_fontsize)
ax2.set_ylim(h_ylim)

ax2.plot(h2_weekly.index, h2_weekly['Višina vodostaja'], color='blue')
# Add mean line for Višina vodostaja
h2_mean = h2['Višina vodostaja'].mean()
ax2.axhline(y=h2_mean, color='blue', linestyle='--', alpha=0.7, label=f'Povprečna višina vodostaja: {h2_mean:.2f} cm')
ax2.tick_params(axis='y', labelcolor='blue', labelsize=tick_fontsize)
ax2.set_title(f"Povprečni tedenski pretok in višina vodostaja\npri postaji Litija na Savi - po prenovi", fontsize=title_fontsize)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, fontsize=tick_fontsize)
ax2.tick_params(axis='x', labelsize=tick_fontsize)

ax2_twin = ax2.twinx()
ax2_twin.set_ylabel('Pretok (m$^3$/s)', color='red', fontsize=label_fontsize)
ax2_twin.set_ylim(q_ylim)
ax2_twin.plot(q2_weekly.index, q2_weekly['Pretok'], color='red')
# Add mean line for Pretok
q2_mean = q2['Pretok'].mean()
ax2_twin.axhline(y=q2_mean, color='red', linestyle='--', alpha=0.7, label=f'Povprečni pretok: {q2_mean:.2f} m³/s')
ax2_twin.tick_params(axis='y', labelcolor='red', labelsize=tick_fontsize)

# Combine both legends into one
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=legend_fontsize)

plt.tight_layout()

In [ ]:
import hidro_and_meteo_helper_functions as sf
import matplotlib.pyplot as plt
import os
# Get precipitation data
precipitation_df = sf.get_meteo_df_for_station(1848).loc['2023-11-03 00:00:00':'2023-11-06 00:00:00']

# Get temperature data
station_id = 1848
station_data_folder = "data/ARSO_podatki_2/20250110/Meteo/temperature/temperature"
files = os.listdir(station_data_folder)
files = [f for f in files if f == f"{station_id}_temp.txt"]
if len(files) == 0:
    raise ValueError(f"No files found for station {station_id}")
elif len(files) > 1:
    print(files)
    raise ValueError(f"Multiple files found for station {station_id}")
file = files[0]
temperature_df = pd.read_csv(os.path.join(station_data_folder, file), sep=',', parse_dates=[0], usecols=[0,1]).rename(columns={'datum': 'date', 't2m':'temperature'})
temperature_df = temperature_df.set_index('date')
temperature_df = temperature_df.sort_index()
temperature_df = temperature_df.loc['2023-11-03 00:00:00':'2023-11-06 00:00:00']

# Set font sizes
label_fontsize = 16
tick_fontsize = 14
title_fontsize = 18
legend_fontsize = 14

# Create figure with two y-axes
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot precipitation data on the primary y-axis
ax1.set_xlabel('Datum', fontsize=label_fontsize)
ax1.set_ylabel('Padavine (cm)', color='blue', fontsize=label_fontsize)
ax1.plot(precipitation_df.index, precipitation_df['rainfall'], color='blue', label='Padavine')
ax1.tick_params(axis='y', labelcolor='blue', labelsize=tick_fontsize)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, fontsize=tick_fontsize)

# Create secondary y-axis and plot temperature data
ax2 = ax1.twinx()
ax2.set_ylabel('Temperatura (°C)', color='red', fontsize=label_fontsize)
ax2.plot(temperature_df.index, temperature_df['temperature'], color='red', label='Temperatura')
ax2.tick_params(axis='y', labelcolor='red', labelsize=tick_fontsize)

# Add title and legend
plt.title('Padavine in temperatura pri postaji Rogla', fontsize=title_fontsize)
#ax1.legend(loc='upper left', fontsize=legend_fontsize)
#ax2.legend(loc='upper right', fontsize=legend_fontsize)

plt.tight_layout()

In [ ]:
import xarray as xr
import cfgrib
import os
import glob
from tqdm import tqdm
from multiprocess import Pool
from concurrent.futures import ThreadPoolExecutor




# Define the path pattern for all GRIB files
base_dir = "data/ALADIN_COMPLETE"
# Find all GRIB files at once (recursively search through all years and months)
grib_files = sorted(glob.glob(os.path.join(base_dir, "*", "*", "ashz_*.grb")))
print(grib_files)

# Function to open and extract 'tp' field from a single file
def extract_tp(filepath):
    import cfgrib
    try:
        try:
            ds = cfgrib.open_datasets(filepath, filter_by_keys={'typeOfLevel': 'surface'},backend_kwargs={
                'filter_by_keys': {'typeOfLevel': 'surface',}, 'indexpath': ''
            }, engine='cfgrib', decode_timedelta = True)[-2][['tp']].load()
            return ds  # Select only total precipitation
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return e
    except Exception as e:
        print(f"Error opening {filepath}: {e}", flush=True)
        return e


#results = [extract_tp(f) for f in tqdm(grib_files)]

print(grib_files[422])
result = extract_tp(grib_files[422])


In [ ]:
result

In [ ]:
# The epygram import is failing because HOME environment variable is not set
# Let's use matplotlib directly to plot the data
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Set global font size for matplotlib
plt.rcParams.update({'font.size': 16})

# Create a figure with a specific projection
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

# Get the data extent from the result
min_lon = result.longitude.min().values
max_lon = result.longitude.max().values
min_lat = result.latitude.min().values
max_lat = result.latitude.max().values

# Extend the view area by 10%
lon_range = max_lon - min_lon
lat_range = max_lat - min_lat
min_lon -= lon_range * 0.05
max_lon += lon_range * 0.05
min_lat -= lat_range * 0.05
max_lat += lat_range * 0.05

# Set the map extent
ax.set_extent([min_lon, max_lon, min_lat, max_lat], crs=ccrs.PlateCarree())

# Plot the precipitation data first so it's underneath the geographic features
im = ax.pcolormesh(result.longitude, result.latitude, result.tp[23].values, 
                  transform=ccrs.PlateCarree())

# Add coastlines and borders for context
ax.coastlines(resolution='10m')
ax.add_feature(cfeature.BORDERS, linestyle='-', alpha=0.5)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
#ax.add_feature(cfeature.LAKES, facecolor='lightblue')
# Add gridlines with longitude and latitude labels
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
# Fix the formatter issue by using cartopy.mpl.gridliner instead
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}

# Add a colorbar
cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Padavine (mm)', fontsize=18)
cbar.ax.tick_params(labelsize=16)

# Add title
plt.title('Vsota vseh napovedanih padavin sistema ALADIN za 27.2.2013', fontsize=20)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr
ds = xr.open_dataset(f"data/ALADIN/aladin_tp_regridded.nc", decode_timedelta=True)
ds.tp.sel(time='2013-02-27 00:00:00')[:24].sum(axis=0).plot()
plt.show()

# The epygram import is failing because HOME environment variable is not set
# Let's use matplotlib directly to plot the data
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Create a figure with a specific projection
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.PlateCarree()})

# Get the data extent from the result
min_lon = ds.lon.min().values
max_lon = ds.lon.max().values
min_lat = ds.lat.min().values
max_lat = ds.lat.max().values

# Set the map extent
ax.set_extent([min_lon, max_lon, min_lat, max_lat], crs=ccrs.PlateCarree())

# Plot the precipitation data first so it's underneath the geographic features
im = ax.pcolormesh(ds.lon, ds.lat, ds.tp.sel(time='2013-02-27 00:00:00')[23].values, 
                  transform=ccrs.PlateCarree())

# Add coastlines and borders for context
ax.coastlines(resolution='10m')
ax.add_feature(cfeature.BORDERS, linestyle='-', alpha=0.5)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
#ax.add_feature(cfeature.LAKES, facecolor='lightblue')
# Add gridlines with longitude and latitude labels
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
# Fix the formatter issue by using cartopy.mpl.gridliner instead
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'size': 18}  # povecaj font
gl.ylabel_style = {'size': 18}  # povecaj font

# Add a colorbar
cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Padavine (mm)', fontsize=18)  # povecaj font
cbar.ax.tick_params(labelsize=16)  # povecaj font

# Add title
plt.title('Vsota vseh napovedanih padavin sistema ALADIN za 27.2.2013', fontsize=22)  # povecaj font

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
ds

In [ ]:
min(result.latitude.values.flatten()), max(result.latitude.values.flatten()),min(result.longitude.values.flatten()), max(result.longitude.values.flatten())

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# Create figure and axes with a specific projection
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Add country borders
ax.add_feature(cfeature.BORDERS, linestyle='-', linewidth=1)

# Set extent to cover the full ALADIN domain
ax.set_extent([9.23, 17.53, 44.103, 48.298], crs=ccrs.PlateCarree())

# Add coastlines
ax.coastlines(resolution='10m')

# Add other features for better visualization
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.LAKES, facecolor='lightblue')
ax.add_feature(cfeature.RIVERS, linewidth=0.5, edgecolor='blue')

# Before plotting, print some information about the data to debug
print(f"Data shape: {result.tp[69].shape}")
print(f"Data min/max: {result.tp[69].min().values} / {result.tp[69].max().values}")

# Try different plotting approaches:

# Approach 1: Explicitly setting vmin and vmax
result.tp[69].plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Purples', 
                  alpha=1.0, add_colorbar=True, vmin=0, 
                  vmax=result.tp[69].max().values)

# If that doesn't work, try approach 2 (comment out approach 1 first):
# import matplotlib.colors as colors
# norm = colors.Normalize(vmin=0, vmax=result.tp[69].max().values * 1.1)
# cs = ax.pcolormesh(result.tp[69].longitude, result.tp[69].latitude, 
#                   result.tp[69].values, transform=ccrs.PlateCarree(),
#                   cmap='Purples', norm=norm, alpha=1.0)
# plt.colorbar(cs, ax=ax, shrink=0.8, label='Precipitation (mm)')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle=':')
gl.top_labels = False
gl.right_labels = False

# Add a title
plt.title('ALADIN Model Domain - Precipitation Data', fontsize=16)

plt.tight_layout()
plt.show()